# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [ ]:


from pathlib import Path
import sys

notebook_dir = Path.cwd()
src_path = (notebook_dir / "../../05_src").resolve()

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))  # insert(0) gives it priority

%load_ext dotenv
%dotenv 


In [ ]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [ ]:
import os
from glob import glob

from utils.logger import get_logger
_logs = get_logger(__name__)

PRICE_DATA = os.getenv("PRICE_DATA")
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)
_logs.info(f'Found {len(parquet_files)} parquet price files.')



 



Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

No it was not necessary.  Dask can also calculate it, however it is better to do it in pandas as it stores the data in memory and allows for faster calculation.  If it wasn't possible to store in memory then Dask would be a good choice.

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
dd_px = dd.read_parquet(parquet_files).set_index("ticker")
dd_shift = (
    dd_px
        .groupby('ticker', group_keys=False)
        .apply(
            lambda x: x.sort_values('Date', ascending = True)
                       .assign(Close_lag_1 = lambda df: df['Close'].shift(1))
                       .assign(Adj_Close_lag_1 = lambda df: df['Adj Close'].shift(1)),
            meta = dd_px._meta.assign(Close_lag_1 = 0.0,
                                     Adj_Close_lag_1 = 0.0))
    )

dd_feat = dd_shift.assign(
    returns = lambda x: x['Close']/x['Close_lag_1'] - 1,
    hi_low_range = lambda x: x['High'] - x['Low']
)





+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [ ]:
df_feat = dd_feat.compute()


df_feat["returns_10_days"] = (
    df_feat.groupby("ticker")["returns"]
    .transform(lambda x: x.rolling(10).mean())
)
df_feat.head(100)

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

No it was not necessary.  Dask can also calculate it, however it is better to do it in pandas as it stores the data in memory and allows for faster calculation.  If it wasn't possible to store in memory then Dask would be a good choice.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.